# ATML PA2 - Task 3 (GRPO) orchestration on Kaggle (2x T4)

This notebook only **orchestrates** the repository scripts; all experiment code lives in `task3_grpo/`.

**Kaggle settings:** Accelerator = *GPU T4 x2*, Internet = *On*. Use **Save Version -> Save & Run All (Commit)**.
The repository is cloned to `/tmp/repo` so that the notebook output contains **only** `task3_artifacts.zip`
(download it with `kaggle kernels output <user>/<notebook> -p <dest>`).

What runs (see `task3_grpo/run_all.sh`):

| GPU 0 | GPU 1 |
|---|---|
| standard 20-update GRPO continuation (K = 4) from the supplied midpoint | equal-generation group-size study on the K = 8 cache (CPU) |
| held-out eval: SFT reference, midpoint, standard | canonical GRPO vs Dr. GRPO forks (8 updates each, per-completion gradient norms) -> eval |

Then `task3_grpo.make_report` builds `report/task3/`.

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/alihasnaink/atml-pa2.git"
BRANCH = "main"
WORKDIR = "/tmp/repo"   # outside /kaggle/working: keeps checkpoints out of the notebook output

import os, subprocess
url = REPO_URL
try:  # optional: private repository
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, WORKDIR], check=True)
else:  # an existing clone (e.g. from an earlier run) is synced to the latest pushed commit
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORKDIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 2. Install the pinned environment

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 3. Download and validate the fixed course assets

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Run all Task 3 experiments
Resumable within a session: re-running this cell skips finished runs/evaluations. Per-GPU logs: `results/task3_grpo/logs/`.

In [ ]:
!bash task3_grpo/run_all.sh

## 5. Quick look at the results

In [ ]:
from IPython.display import Markdown, Image, display
for t in ["heldout_all_conditions", "standard_continuation_compute", "signal_and_sample_efficiency",
          "group_size_equal_generations", "group_size_by_difficulty", "normalization_comparison", "paired_vs_midpoint"]:
    p = f"report/task3/tables/{t}.md"
    if os.path.exists(p):
        display(Markdown(open(p).read()))
for f in ["standard_continuation_diagnostics", "group_size_by_difficulty", "prompt_difficulty_vs_std",
          "normalization_comparison"]:
    p = f"report/task3/figures/{f}.png"
    if os.path.exists(p):
        display(Image(p))

## 6. Package the artifacts
Unzip `task3_artifacts.zip` at the repository root. It contains `results/task3_grpo/`, `report/task3/` and the LoRA adapters in `outputs/task3_grpo/` (git-ignored; keep `outputs/task3_grpo/standard`, Task 4 uses it).

In [ ]:
!cd /tmp/repo && zip -qr /kaggle/working/task3_artifacts.zip results/task3_grpo report/task3 outputs/task3_grpo
!ls -lh /kaggle/working/